In [1]:
import yfinance as yf
import numpy as np
import pandas as pd

In [55]:
px = yf.download(["EWA", "EWC"], start="2006-04-26", end="2012-04-26", auto_adjust=False)["Close"].dropna()
x, y = px["EWA"].values , px["EWC"].values

[*********************100%***********************]  2 of 2 completed


### The Kalman filter

In [56]:
def kalman_beta(x, y, delta=1e-4, ve=1e-3):
    X = np.column_stack([x, np.ones(len(x))]) # -> [x, 1]
    n = len(y)
    beta = np.zeros((n, 2))
    e = np.full(n, np.nan)
    q = np.full(n, np.nan)
    b,p = np.zeros(2), np.zeros((2,2))
    vw = delta / (1-delta) * np.eye(2)
    for t in range(n):
        r = p + vw if t > 0 else np.zeros((2,2))
        xt = X[t]
        q[t] = xt @ r @ xt + ve      # variance of prediction error
        e[t] = y[t] - xt @ b         # surprise: actual - predicted
        k = r @ xt / q[t]            # gain: how much to trust the surprise
        b = b + k * e[t]             # update beta
        p = r - np.outer(k, xt) @ r  # update uncertainty
        beta[t] = b

    return beta, e, q

In [57]:
beta, e , Q = kalman_beta(x, y)
s = np.sqrt(Q)

### Signals

In [58]:
def signals(e, s, exit_=1.0):
    e, s = pd.Series(e), pd.Series(s)
    long, short = pd.Series(np.nan, e.index), pd.Series(np.nan, e.index)
    long[e < -s] = 1;  long[e >= -exit_ * s] = 0
    short[e > s] = -1; short[e <= exit_ * s] = 0
    return long.ffill().fillna(0) + short.ffill().fillna(0)

units = signals(e, s, exit_=1.0)   # exit_=1 reproduces Chan's logic exactly

### P&L

In [67]:
def backtest(units, beta, x, y):
    u = units.shift(1).values                  # hold yesterday's position
    b = np.roll(beta[:, 0], 1)                 # yesterday's hedge ratio
    dy, dx = np.diff(y, prepend=np.nan), np.diff(x, prepend=np.nan)
    pnl   = u * (dy - b * dx)                  # long 1 EWC, short b EWA
    gross = np.abs(u) * (np.roll(y, 1) + np.abs(b) * np.roll(x, 1))
    ret = np.divide(pnl, gross, out=np.zeros_like(pnl), where=gross > 0)
    apr = (1 + ret).prod() ** (252 / len(ret)) - 1
    sharpe = np.sqrt(252) * ret.mean() / ret.std()
    return ret, apr, sharpe

ret, apr, sharpe = backtest(units, beta, x, y)
print(f"APR={apr:.4f} Sharpe={sharpe:.2f}")

APR=0.2054 Sharpe=2.15


In [68]:
units = signals(e, s, exit_=0.0)
ret, apr, sharpe = backtest(units, beta, x, y)
print(f"APR={apr:.4f} Sharpe={sharpe:.2f}")

APR=0.2054 Sharpe=2.15


In [69]:
print("position changes:", int((units.diff().abs() > 0).sum()))

position changes: 708


### net cost stats

In [70]:
def net_stats(ret, units, c=0.0005):
    net = ret - c * units.diff().abs().fillna(0).values
    apr = (1 + net).prod() ** (252 / len(net)) - 1
    return apr, np.sqrt(252) * net.mean() / net.std()

In [71]:
for ex in (1.0, 0.0):
    u = signals(e, s, exit_=ex)
    r, _, _ = backtest(u, beta, x, y)
    print(f"exit_={ex} changes={int((u.diff().abs()>0).sum())}")
    for c in (0, 0.0005, 0.001):
        a, sh = net_stats(r, u, c)
        print(f"  cost={c*1e4:.0f}bps APR={a:.4f} Sharpe={sh:.2f}")

exit_=1.0 changes=820
  cost=0bps APR=0.1709 Sharpe=1.96
  cost=5bps APR=0.0793 Sharpe=1.00
  cost=10bps APR=-0.0052 Sharpe=-0.03
exit_=0.0 changes=708
  cost=0bps APR=0.2054 Sharpe=2.15
  cost=5bps APR=0.1166 Sharpe=1.34
  cost=10bps APR=0.0342 Sharpe=0.46
